In [23]:
from pathlib import Path
import heapq
import struct
import zlib
from collections import Counter

import numpy as np
from PIL import Image

ROOT = Path("Bird Speciees Dataset")   

data = []
paths = []
for path in sorted(ROOT.glob("*")):
    if path.suffix.lower() in [".jpg", ".jpeg"]:
        with Image.open(path) as im:
            data.append(np.asarray(im.convert("RGB")))
        paths.append(path)

print(f"Loaded {len(data)} images")
if data:
    print("Example:", paths[0], data[0].shape, data[0].dtype)

Loaded 0 images


### Huffman

In [24]:
def huffman_code_lengths(freqs):
    """freqs: dict symbol(int 0..255) -> count. Returns dict symbol -> bit length."""
    symbols = [s for s, f in freqs.items() if f > 0]
    if not symbols:
        return {}
    if len(symbols) == 1:
        return {symbols[0]: 1}   # single-symbol edge case: 1 bit per symbol

    # Deterministic tie-breaking: (freq, insertion order, node)
    heap = [(freqs[s], i, s) for i, s in enumerate(symbols)]
    heapq.heapify(heap)
    tie = len(symbols)

    while len(heap) > 1:
        f1, _, n1 = heapq.heappop(heap)
        f2, _, n2 = heapq.heappop(heap)
        heapq.heappush(heap, (f1 + f2, tie, (n1, n2)))
        tie += 1

    root = heap[0][2]

    lengths = {}
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if isinstance(node, int):
            lengths[node] = depth
        else:
            left, right = node
            stack.append((left, depth + 1))
            stack.append((right, depth + 1))
    return lengths


def canonical_codes(lengths):
    """lengths: dict symbol -> length. Returns dict symbol -> (code_int, length).
    Canonical Huffman: sort by (length, symbol), assign incrementally."""
    items = sorted(lengths.items(), key=lambda kv: (kv[1], kv[0]))
    codes = {}
    code = 0
    prev_len = 0
    for sym, length in items:
        code <<= (length - prev_len)
        codes[sym] = (code, length)
        code += 1
        prev_len = length
    return codes

### Bit-level IO

In [25]:
def encode_symbols(symbols, codes):
    """symbols: iterable of ints. codes: dict sym -> (code, length). MSB-first packing."""
    out = bytearray()
    acc = 0
    nbits = 0
    for s in symbols:
        code, length = codes[s]
        acc = (acc << length) | code
        nbits += length
        while nbits >= 8:
            nbits -= 8
            out.append((acc >> nbits) & 0xFF)
    if nbits > 0:
        out.append((acc << (8 - nbits)) & 0xFF)
    return bytes(out)


def decode_bits(payload, codes, n_symbols):
    """Inverse of encode_symbols. Reads exactly n_symbols symbols."""
    by_len = {}
    for sym, (code, length) in codes.items():
        by_len.setdefault(length, {})[code] = sym

    out = bytearray(n_symbols)
    acc = 0
    nbits = 0
    bit_pos = 0
    total_bits = len(payload) * 8

    for i in range(n_symbols):
        while nbits not in by_len or acc not in by_len[nbits]:
            if bit_pos >= total_bits:
                raise ValueError(f"ran out of bits at symbol {i}/{n_symbols}")
            byte = payload[bit_pos >> 3]
            bit = (byte >> (7 - (bit_pos & 7))) & 1
            acc = (acc << 1) | bit
            nbits += 1
            bit_pos += 1
        out[i] = by_len[nbits][acc]
        acc = 0
        nbits = 0
    return bytes(out)

### Container

In [26]:
MAGIC = b"HFM1"

def pack_header(dtype_str, shape, original_n_bytes, code_lengths, crc):
    assert len(code_lengths) == 256
    dtype_bytes = dtype_str.encode("ascii")
    out = bytearray()
    out += MAGIC                      # 4
    out.append(len(dtype_bytes))      # 1
    out += dtype_bytes                # dtype_len
    out.append(len(shape))            # 1
    for d in shape:
        out += struct.pack("<I", d)   # 4 * ndim
    out += struct.pack("<Q", original_n_bytes)   # 8
    out += bytes(code_lengths)        # 256
    out += struct.pack("<I", crc)     # 4
    return bytes(out)


def unpack_header(buf):
    if buf[:4] != MAGIC:
        raise ValueError("bad magic")
    pos = 4
    dtype_len = buf[pos]; pos += 1
    dtype_str = buf[pos:pos + dtype_len].decode("ascii"); pos += dtype_len
    ndim = buf[pos]; pos += 1
    shape = []
    for _ in range(ndim):
        shape.append(struct.unpack_from("<I", buf, pos)[0]); pos += 4
    shape = tuple(shape)
    original_n_bytes = struct.unpack_from("<Q", buf, pos)[0]; pos += 8
    code_lengths = list(buf[pos:pos + 256]); pos += 256
    crc = struct.unpack_from("<I", buf, pos)[0]; pos += 4
    payload = buf[pos:]
    return dtype_str, shape, original_n_bytes, code_lengths, crc, payload

## Interface

In [27]:
def compress(data, settings=None):
    """data: numpy array. Returns bytes (self-contained)."""
    arr = np.ascontiguousarray(data)
    raw = arr.tobytes()
    n = len(raw)

    if n == 0:
        code_lengths = [0] * 256
        crc = zlib.crc32(b"") & 0xFFFFFFFF
        return pack_header(str(arr.dtype), arr.shape, 0, code_lengths, crc)

    freqs = Counter(raw)
    lengths = huffman_code_lengths(freqs)
    codes = canonical_codes(lengths)
    payload = encode_symbols(raw, codes)
    crc = zlib.crc32(raw) & 0xFFFFFFFF
    code_lengths = [lengths.get(s, 0) for s in range(256)]

    header = pack_header(str(arr.dtype), arr.shape, n, code_lengths, crc)
    return header + payload


def decompress(compressed, settings_or_metadata=None):
    """compressed: bytes. Returns numpy array with the original shape and dtype."""
    dtype_str, shape, original_n_bytes, code_lengths, crc, payload = unpack_header(compressed)

    if original_n_bytes == 0:
        raw = b""
    else:
        lengths = {s: L for s, L in enumerate(code_lengths) if L > 0}
        if not lengths:
            raise ValueError("no code lengths in header")
        codes = canonical_codes(lengths)
        raw = decode_bits(payload, codes, original_n_bytes)

    if (zlib.crc32(raw) & 0xFFFFFFFF) != crc:
        raise ValueError("CRC mismatch — payload corrupted")

    arr = np.frombuffer(raw, dtype=np.dtype(dtype_str)).reshape(shape)
    return arr.copy()   # return a writable array

### Byte accounting helper

In [28]:
def byte_report(arr, compressed):
    orig = arr.nbytes
    comp = len(compressed)
    return {
        "original_bytes": orig,
        "compressed_bytes": comp,
        "ratio": orig / comp if comp else float("nan"),
        "bits_per_input_byte": 8 * comp / orig if orig else float("nan"),
    }


def print_report(label, arr, compressed):
    r = byte_report(arr, compressed)
    print(f"{label:30s} "
          f"orig={r['original_bytes']:>8d}  "
          f"comp={r['compressed_bytes']:>8d}  "
          f"ratio={r['ratio']:.3f}  "
          f"bits/byte={r['bits_per_input_byte']:.3f}")

###  Round-trip tests (shape, dtype, empty, tiny, unseen)

In [29]:
def roundtrip(arr, label=""):
    comp = compress(arr)
    dec = decompress(comp)
    assert dec.shape == arr.shape, f"shape mismatch {label}: {dec.shape} vs {arr.shape}"
    assert dec.dtype == arr.dtype, f"dtype mismatch {label}: {dec.dtype} vs {arr.dtype}"
    assert np.array_equal(dec, arr), f"data mismatch {label}"
    return comp

rng = np.random.default_rng(0)

tests = [
    ("empty uint8",       np.zeros((0,), dtype=np.uint8)),
    ("tiny 1 elem",       np.array([42], dtype=np.uint8)),
    ("tiny 1x1 float32",  np.array([[3.14]], dtype=np.float32)),
    ("1xN int16",         np.arange(16, dtype=np.int16)),
    ("2D uint8 ramp",     (np.arange(256, dtype=np.uint8).reshape(16, 16))),
    ("3D uint8 uniform",  rng.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)),
    ("float32 noise",     rng.standard_normal((32, 32)).astype(np.float32)),
    ("all same symbol",   np.full((100,), 7, dtype=np.uint8)),
]

for label, arr in tests:
    comp = roundtrip(arr, label)
    print_report(label, arr, comp)

empty uint8                    orig=       0  comp=     283  ratio=0.000  bits/byte=nan
tiny 1 elem                    orig=       1  comp=     284  ratio=0.004  bits/byte=2272.000
tiny 1x1 float32               orig=       4  comp=     290  ratio=0.014  bits/byte=580.000
1xN int16                      orig=      32  comp=     295  ratio=0.108  bits/byte=73.750
2D uint8 ramp                  orig=     256  comp=     543  ratio=0.471  bits/byte=16.969
3D uint8 uniform               orig=     192  comp=     459  ratio=0.418  bits/byte=19.125
float32 noise                  orig=    4096  comp=    4041  ratio=1.014  bits/byte=7.893
all same symbol                orig=     100  comp=     296  ratio=0.338  bits/byte=23.680


### Test on data

In [30]:
if data:
    # Train/develop on the first image
    train_img = data[0]
    comp = roundtrip(train_img, paths[0].name)
    print_report(paths[0].name, train_img, comp)

    # Unseen file: last image, not used during development
    if len(data) > 1:
        unseen = data[-1]
        comp_u = roundtrip(unseen, paths[-1].name + " (unseen)")
        print_report(paths[-1].name + " (unseen)", unseen, comp_u)

## Baseline comparison 

In [31]:
def baseline_zlib(arr, level=6):
    raw = np.ascontiguousarray(arr).tobytes()
    return zlib.compress(raw, level=level)


if data:
    img = data[0]
    ours = compress(img)
    base = baseline_zlib(img, level=6)
    print("Baseline zlib-6 vs our Huffman on:", paths[0].name)
    print_report("ours (Huffman + header)", img, ours)
    print_report("baseline (zlib-6)", img, base)
    print(f"zlib payload only size: {len(base)} bytes "
          f"(zlib has its own ~6 byte header)")

### Failure case

In [32]:
# 1) Uniform random bytes: entropy is maximal, Huffman cannot shrink it.
random_bytes = rng.integers(0, 256, size=100_000, dtype=np.uint8)
c = compress(random_bytes)
print_report("uniform random 100k", random_bytes, c)
# Expect: ratio slightly < 1 because of the ~290-byte header.

# 2) Already-compressed data (JPEG file on disk): same situation.
if paths:
    jpeg_bytes = np.frombuffer(paths[0].read_bytes(), dtype=np.uint8)
    c2 = compress(jpeg_bytes)
    print_report("raw JPEG bytes", jpeg_bytes, c2)
    # Expect: ratio <= 1, i.e. no useful compression.

# 3) Tiny inputs: header dominates.
tiny = np.array([1, 2, 3], dtype=np.uint8)
print_report("3-byte input", tiny, compress(tiny))
# Expect: ratio << 1 (expansion). Document this as "do not use Huffman on tiny inputs".

uniform random 100k            orig=  100000  comp=  100283  ratio=0.997  bits/byte=8.023
3-byte input                   orig=       3  comp=     284  ratio=0.011  bits/byte=757.333
